In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install sentence-transformers openpyxl scikit-learn

In [6]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/LLM_odev2_veriler.xlsx')

for i in range(len(df)):
  check_valid = len(str(df.loc[i, 'CosmosLLM düşünme süreci']))
  if check_valid <= 20:
    df.drop(i, inplace=True)

In [7]:
import random as r

l = [1,2,3]
for i in range(3):
  r.seed("AIHW2")
  r.choice(l)

def split_Data():
  testing_place = r.sample(range(len(df)), 1000)
  testing_rows = df.iloc[testing_place]


split_Data()

In [8]:
score_change = {
    'çok iyi': 5,
    'iyi': 4,
    'orta': 3,
    'kötü': 2,
    'çok kötü': 1
}

def map_score(value):
    if isinstance(value, str):
        return score_change.get(value, value)
    return value

df['Değerlendirme Puanınız'] = df['Değerlendirme Puanınız'].apply(map_score)

print(df["Değerlendirme Puanınız"])

0        4
1        3
2        5
3        4
4        2
        ..
12129    3
12130    5
12131    3
12132    3
12133    2
Name: Değerlendirme Puanınız, Length: 10990, dtype: int64


In [9]:
#S, D, C, score = df['Sorunuz'], df['CosmosLLM düşünme süreci'], df['CosmosLLM cevabı'], df['Değerlendirme Puanınız']
#print(df.columns)
df.columns = ['S', 'D', 'C', 'score']
print(df.columns)

Index(['S', 'D', 'C', 'score'], dtype='object')


In [10]:
config1 = df["S"].tolist()
config2 = df["D"].tolist()
config3 = df["C"].tolist()
config4 = (df["S"].astype(str) + "\n" + df["D"].astype(str)).tolist()
config5 = (df["D"].astype(str) + "\n" + df["C"].astype(str)).tolist()

In [13]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import os # Import the os module

model = SentenceTransformer("ytu-ce-cosmos/turkish-e5-large", device="cuda")

# Ensure all elements in config2 are string
config_str = [str(x) for x in config5]

embeddings = model.encode(config_str, batch_size=32, show_progress_bar=True)

# Define the directory path
output_dir = "/content/drive/MyDrive/AI_HW/"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save the embeddings
np.save(os.path.join(output_dir, "ce_cosmos_config5.npy"), embeddings)

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/344 [00:00<?, ?it/s]